# Gradient Boosting (XGBoost) — Telco Churn

XGBoost com Optuna HPO (maximiza KS no val). CUDA habilitado via `device='cuda'` (XGBoost 3.x).

In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn import metrics as skm
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'XGBoost version: {xgb.__version__}')

XGBoost version: 3.0.0


## 1. Carregar dados

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

Xtr_np  = X_train.values;  ytr_np  = y_train.values.astype(int)
Xvl_np  = X_val.values;    yvl_np  = y_val.values.astype(int)
Xte_np  = X_test.values;   yte_np  = y_test.values.astype(int)

Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)


## 2. Optuna HPO (40 trials, maximiza KS no val)

In [3]:
def ks_score(y_true, y_prob):
    fpr, tpr, _ = skm.roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'random_state': RANDOM_STATE,
        'device': 'cuda',
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=30, verbosity=0)
    model.fit(Xtr_np, ytr_np,
              eval_set=[(Xvl_np, yvl_np)],
              verbose=False)
    probs = model.predict_proba(Xvl_np)[:, 1]
    return ks_score(yvl_np, probs)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=40, show_progress_bar=True)
print(f'Best KS (val): {study.best_value:.4f}')
print('Best params:', study.best_params)

  0%|          | 0/40 [00:00<?, ?it/s]

Best KS (val): 0.5456
Best params: {'n_estimators': 976, 'max_depth': 3, 'learning_rate': 0.029830872476130286, 'subsample': 0.9918585998724917, 'colsample_bytree': 0.881434643981817, 'min_child_weight': 2}


## 3. Retreinar com melhores parâmetros

In [4]:
bp = study.best_params
best_model = xgb.XGBClassifier(
    **bp,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    device='cuda',
    early_stopping_rounds=50,
    verbosity=1,
)
best_model.fit(Xtr_np, ytr_np,
               eval_set=[(Xvl_np, yvl_np)],
               verbose=50)
print('Best iteration:', best_model.best_iteration)

[0]	validation_0-logloss:0.68423
[50]	validation_0-logloss:0.52824
[100]	validation_0-logloss:0.50287
[150]	validation_0-logloss:0.49522
[200]	validation_0-logloss:0.49250
[250]	validation_0-logloss:0.49123
[300]	validation_0-logloss:0.49058
[350]	validation_0-logloss:0.48998
[400]	validation_0-logloss:0.48967
[450]	validation_0-logloss:0.48961
[468]	validation_0-logloss:0.48950
Best iteration: 418


## 4. Avaliar no teste + salvar artefatos

In [5]:
probs = best_model.predict_proba(Xte_np)[:, 1]
preds = (probs >= 0.5).astype(int)

scores = compute_metrics(yte_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/gradient_boosting', exist_ok=True)
joblib.dump(best_model, 'results/gradient_boosting/model.pkl')

save_results('gradient_boosting', yte_np, preds, probs, scores)

Scores no teste:
  accuracy               0.7455
  balanced_accuracy      0.7698
  precision              0.5122
  recall                 0.8214
  f1                     0.6310
  auroc                  0.8487
  ks                     0.5460
Saved → results/gradient_boosting
